In [1]:
import mlflow
import os
import pickle
import pandas as pd
import uuid

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline

In [11]:
YEAR = 2021
MONTH = 2
TAXI_TYPE = 'green'
!mkdir output
!mkdir output/$TAXI_TYPE

input_file = f'https://d37ci6vzurychx.cloudfront.net/trip-data/{TAXI_TYPE}_tripdata_{YEAR:04d}-{MONTH:02d}.parquet'
output_file = f'output/{TAXI_TYPE}/{YEAR:04d}-{MONTH:02d}.parquet'

mkdir: cannot create directory ‘output’: File exists
mkdir: cannot create directory ‘output/green’: File exists


In [3]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI','http://localhost:5000')
RUN_ID = os.getenv('RUN_ID', '38ee7005f2d64eff8f1e2edad80cd244')

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [8]:
def generate_uuids(n):
    ride_ids = []
    
    for i in range(n):
        ride_ids.append(str(uuid.uuid4()))

    return ride_ids


def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    df['ride_id'] = generate_uuids(len(df))
    
    return df


def prepare_dictionaries(df: pd.DataFrame):
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts


def load_model(run_id):
    # Not hosted in S3 so we need to manually load the model from the run
    logged_model = f"runs:/{run_id}/model"
    model = mlflow.pyfunc.load_model(logged_model)

    return model


def compute_results(df, y_pred, run_id):
    df_result = pd.DataFrame()
    
    df_result['ride_id'] = df['ride_id']
    df_result['lpep_pickup_datetime'] = df['lpep_pickup_datetime']
    df_result['PULocationID'] = df['PULocationID']
    df_result['DOLocationID'] = df['DOLocationID']
    df_result['actual_duration'] = df['duration']
    df_result['predicted_duration'] = y_pred
    df_result['diff'] = df_result['actual_duration'] - df_result['predicted_duration']
    df_result['model_version'] = run_id

    return df_result


def apply_model(input_file, run_id, output_file):
    df = read_dataframe(input_file)
    dicts = prepare_dictionaries(df)
    
    model = load_model(run_id)
    y_pred = model.predict(dicts)

    computed_results = compute_results(df, y_pred, run_id)

    computed_results.to_parquet(output_file, index=False)

In [12]:
apply_model(input_file=input_file, run_id=RUN_ID, output_file=output_file)

2026/07/23 15:22:02 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - aiohttp (current: 3.14.1, required: aiohttp==3.14.2; sys_platform == "linux")
 - alembic (current: 1.18.4, required: alembic==1.18.5; sys_platform == "linux")
 - anyio (current: 4.13.0, required: anyio==4.14.2; sys_platform == "linux")
 - certifi (current: 2026.4.22, required: certifi==2026.7.22; sys_platform == "linux")
 - cffi (current: 2.0.0, required: cffi==2.1.0; platform_python_implementation != "PyPy" and sys_platform == "linux")
 - charset-normalizer (current: 3.4.7, required: charset-normalizer==3.4.9; sys_platform == "linux")
 - click (current: 8.3.3, required: click==8.4.2; sys_platform == "linux")
 - cryptography (current: 47.0.0, required: cryptography==48.0.1; sys_platform == "linux")
 - databricks-sdk (current: 0.106.0, required: databricks-sdk==0.122.0; sys_platform == "linux")
 - docker (current: 7.1.0, req